In [1]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model, cross_validate_model, tune_xgboost

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "all" # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (1607, 41)


/Users/sharanashivanand/git_repos/dsr/cropspectai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

print("Features:")
print(X.columns.tolist())

Targets:
['cab']
Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


In [3]:
top_indices = ['mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343'] 
# top_indices = ['mndvi_as7343', 'sipi_as7343']
# top_indices = ['mndvi_as7343']
# top_indices = ['red_edge_green_nd']
# top_indices = ['red_green_nd'] # 1
# top_indices = ['psri_as7343'] # 2
# top_indices = ['datt1_as7343']   
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())
full_list = AS7343_CHANNELS + top_indices
X = X[full_list].copy()
features = X.columns.tolist()

print("Features:")
print(features)
print(X.shape)

df = pd.concat([X, y], axis = 1).copy()
print(df.shape)

Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343']
(1607, 19)
(1607, 20)


## Hyperspectra up to 1000nm only

In [4]:
if spectral_length == "1000":
    print(X.shape)
    X_1000 = X.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y.shape)

    df = pd.concat([X_1000, y], axis = 1).copy()
    print(df.shape)

In [5]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every feature and every target
# --------------------------------------------------

# for feature in features:
for target in targets:

    print("\n" + "=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)

    # IMPORTANT:
    # Start from the original dataframe for every target
    df_target = preprocessing.drop_duplicates(
        df,
        target
    ).copy()

    # Split into features and targets AFTER target-specific
    # duplicate handling
    X, y = preprocessing.split_dataset_into_X_y(df_target, target)

    scoring = "r2"

    #------------------------only k fold cv score-------------------------

    # # Create a fresh model for every target
        # model_xgb = XGBRegressor(
        #     n_estimators=1000,
        #     max_depth=6,
        #     learning_rate=0.01,
        #     subsample=0.8,
        #     colsample_bytree=1.0,
    
        #     tree_method="hist",
    
        #     objective="reg:squarederror",
        #     eval_metric="rmse",
    
        #     n_jobs=-1,
        #     random_state=42,
        # )

    # cv_scores = cross_validate_model(model_xgb, X, y, n_splits=7, scale=False, scoring=scoring)

    #------------------------------------------------------------------

    #---------------------optuna study-----------------------

    study = tune_xgboost(
    X,
    y,
    n_splits=7,
    n_trials=50,
    scale=False,    
    scoring=scoring,
    random_state=42,
    )

[I 2026-10-12 17:01:04,810] A new study created in memory with name: no-name-e2720da0-83da-427c-891c-2b65f37781a5



TARGET: cab
(Before) Shape of dataframe: (1607, 20)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 20)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 20)


[I 2026-10-12 17:01:15,267] Trial 0 finished with value: 0.5701648592948914 and parameters: {'use_l1': False, 'n_estimators': 1000, 'learning_rate': 0.01, 'max_depth': 6, 'min_child_weight': 1.0, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_lambda': 1.0}. Best is trial 0 with value: 0.5701648592948914.
[I 2026-10-12 17:01:21,723] Trial 1 finished with value: 0.5576030101094928 and parameters: {'use_l1': True, 'reg_alpha': 0.45705630998014507, 'n_estimators': 1300, 'learning_rate': 0.007979118876474874, 'max_depth': 3, 'min_child_weight': 1.218418650222176, 'subsample': 0.9464704583099741, 'colsample_bytree': 0.8404460046972835, 'reg_lambda': 3.470266988650414}. Best is trial 0 with value: 0.5701648592948914.
[I 2026-10-12 17:01:24,698] Trial 2 finished with value: 0.5401790738105774 and parameters: {'use_l1': True, 'reg_alpha': 1.452824663751602, 'n_estimators': 600, 'learning_rate': 0.008620446097910767, 'max_depth': 3, 'min_child_weight': 2.8145092716060645, 'subsample': 0.8099025


🏆 Best trial:
   tree_method = hist
   objective = reg:squarederror
   eval_metric = rmse
   n_jobs = -1
   random_state = 42
   n_estimators = 600
   learning_rate = 0.016646
   max_depth = 7
   min_child_weight = 8.907319
   subsample = 0.867761
   colsample_bytree = 0.972981
   reg_alpha = 0.000000
   reg_lambda = 22.529137
   Mean CV R² = 0.5783


In [ ]:
#------------------------only k fold cv score-------------------------

# print(f"CV ({scoring}) scores: {cv_scores}")

# print(f"Mean CV ({scoring}) score: {cv_scores.mean()}")

## Save optuna-based best model

In [9]:
import json
import joblib
from xgboost import XGBRegressor
from cropspectai.modeling.train import build_pipeline

In [10]:
best_trial = study.best_trial
best_params = best_trial.user_attrs["model_params"]

# Fit the winning configuration on all development data (X and y)
best_pipeline = build_pipeline(
    XGBRegressor(**best_params),
    scale=False,
)
best_pipeline.fit(X, y) 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['F1','F2','FZ',...,'red_green_nd','datt1_as7343','mcari_osavi_as7343']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None


### save best pipeline

In [11]:
file_name = f"xgb_v_01.joblib"
output_path = config.MODELS_DIR / file_name

# Save the fitted pipeline, including preprocessing.
joblib.dump(best_pipeline, output_path)

['/Users/sharanashivanand/git_repos/dsr/cropspectai/models/xgb_v_01.joblib']

### save best parameters

In [12]:
# Save parameters and CV results.
best_results = {
    "target": target,
    "scoring": scoring,
    "score_units": (
        "percent"
        if scoring == "neg_mean_absolute_percentage_error"
        else "R2"
    ),
    "mean_cv_score": study.best_value,
    "std_cv_score": best_trial.user_attrs["score_std"],
    "fold_scores": best_trial.user_attrs["fold_scores"],
    "best_params": best_params,
    "features": X.columns.tolist(),
    "n_samples": len(X),
}

with (config.MODELS_DIR / "xgb_best_parameters_v_01.json").open("w") as file:
    json.dump(best_results, file, indent=2)

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets